# Jodhpur House Price Prediction
## 1. Data Cleaning & Exploratory Data Analysis (EDA)

This notebook processes the raw data, handles missing values (imputation) and outliers, visualizes the distribution, and exports a cleaned dataset for model training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

### Step 1: Load Raw Data

In [ ]:
# Load the dirty dataset
df = pd.read_csv('jodhpur_housing_dirty.csv')
print("=== RAW DATASET LOADED ===")
print(f"Initial shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### Step 2: Drop Irrelevant Columns

In [ ]:
# Drop unwanted columns that do not contribute to price prediction
cols_to_drop = ['Distance_Airport', 'Distance_Railway_Station', 'Distance_City_Center', 'Furnishing_Status', 'Total_Floors', 'Floor_No']
dropped = [col for col in cols_to_drop if col in df.columns]
df.drop(columns=dropped, inplace=True)

print(f"Dropped columns: {dropped}")
print(f"New shape: {df.shape}")

### Step 3: Missing Value Imputation
We check how many missing values exist, and impute them (replace them) using the Median for numbers and Mode for categories.

In [ ]:
# Ensure Area_Size is numeric
if 'Area_Size' in df.columns and df['Area_Size'].dtype == 'object':
    df['Area_Size'] = pd.to_numeric(df['Area_Size'], errors='coerce')

print("=== MISSING VALUES BEFORE IMPUTATION ===")
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])

imputation_log = []

# Handle missing values
for col in df.columns:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        if df[col].dtype in ['int64', 'float64']:
            val = df[col].median()
            df[col] = df[col].fillna(val)
            imputation_log.append(f"{col}: Imputed {missing_count} missing values with Median ({val})")
        else:
            val = df[col].mode()[0]
            df[col] = df[col].fillna(val)
            imputation_log.append(f"{col}: Imputed {missing_count} missing values with Mode ({val})")

print("\n=== IMPUTATION SUMMARY ===")
for log in imputation_log:
    print(log)

print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")

### Step 4: Handle Outliers (IQR Method)
Extreme property prices or sizes can skew the AI. We cap them using the Interquartile Range (IQR).

In [ ]:
# Cap outliers using 1.5 * IQR
continuous_cols = ['Area_Size', 'Price', 'Property_Age']

print("=== OUTLIER CAPPING SUMMARY ===")
for col in continuous_cols:
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        
        df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
        df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
        
        if outliers_count > 0:
            print(f"{col}: Capped {outliers_count} extreme outliers.")

### Step 5: Format Categorical Data
Convert Yes/No text into 1/0 numbers for the machine learning algorithms.

In [ ]:
# Convert Yes/No amenities to 1/0
yes_no_columns = ['Gym', 'Swimming_Pool', 'Market', 'School', 'Hospital', 
                  'Park', 'Mall', 'Security', 'Lift', 'Parking', 'Power_Backup']

converted = 0
for col in yes_no_columns:
    if col in df.columns:
        df[col] = df[col].map({'Yes': 1, 'No': 0, 1: 1, 0: 0})
        df[col] = df[col].fillna(0).astype(int)
        converted += 1
        
print(f"Successfully converted {converted} text columns into binary (1/0) format.")
df.head()

### Step 6: Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['Price'], bins=50, kde=True)
plt.title('Distribution of Property Prices (After Capping Outliers)')
plt.xlabel('Price')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='Area_Name', y='Price', data=df)
plt.title('Price Distribution by Area')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Correlation matrix for numerical features
numeric_df = df.select_dtypes(include=['int64', 'float64', 'int32'])
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

### Step 7: Save Cleaned Dataset

In [ ]:
df.to_csv('jodhpur_housing_cleaned.csv', index=False)
print("=== FINAL DATASET EXPORTED ===")
print(f"Cleaned dataset saved as 'jodhpur_housing_cleaned.csv'")
print(f"Final shape: {df.shape[0]} rows, {df.shape[1]} columns ready for Model Training.")